# 03. Térbeli Elemzés és Interaktív Hőtérképek

**Cél**: Kőbánya ingatlanpiaci értéktérképének vizuális megjelenítése a garantált pontos GIS koordinátával rendelkező lakásokon (N = 296 db).

---

### 📖 Miért a lokáció az ingatlanpiac legfőbb törvénye?
Egy lakás belső tereit át lehet alakítani, de **a fizikai elhelyezkedése megváltoztathatatlan**.
Ebben a fejezetben a pontos GPS koordinátákkal rendelkező hirdetéseket vetítjük térképre, hogy feltárjuk az árfoltokat, a tömegközlekedési tengelyeket és a területi egyenlőtlenségeket.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Garantált pontos koordinátákkal rendelkező eladó hirdetések száma: {len(df_pontos)} db.")

Garantált pontos koordinátákkal rendelkező eladó hirdetések száma: 254 db.


### 1. Interaktív Ingatlan Ponttérkép

**📌 Mit látunk a térképen?**
A 296 db garantált pontos ingatlan térbeli elhelyezkedését:
- **A pontok színe**: A négyzetméterárat mutatja a lila (alacsonyabb árszint) árnyalatoktól a sárga és zöld (1.3 - 1.5 millió Ft/m² feletti prémium) pontokig.
- **A pontok mérete**: A lakás alapterületével arányos.
- **Interaktivitás**: Az egérrel a pontok fölé állva megjelenik a pontos cím, az ár és a városrész.

**🔍 Térbeli mintázatok**:
A Hungária körúthoz és Zuglóhoz közeli északi sáv (Ligettelek, Városközpont) világosabb, míg a külső ipari-vasúti folyosók mentén a sötétebb pontok dominálnak.

In [2]:
n_pontos = len(df_pontos)
in_mazsa_10p = int((df_pontos['mazsa_10p_seta'] == 1).sum()) if 'mazsa_10p_seta' in df_pontos.columns else 0
in_metro_10p = int((df_pontos['metro_10p_seta'] == 1).sum()) if 'metro_10p_seta' in df_pontos.columns else 0
atlag_belvaros_km = df_pontos['tavolsag_belvaros_halozati_m'].mean() / 1000 if 'tavolsag_belvaros_halozati_m' in df_pontos.columns else 6.5
med_nm_ar_pontos = df_pontos['nm_ar_huf'].median()

kpi_cards = [
    ("Garantált Pontos Minta", f"{n_pontos} db", "100% valós koordináta (eladó)", "#1e3a8a"),
    ("Mázsa tér 10p Séta", f"{in_mazsa_10p} db", f"{in_mazsa_10p/n_pontos*100:.1f}% lefedettség", "#ec4899"),
    ("Metró 10p Séta", f"{in_metro_10p} db", f"{in_metro_10p/n_pontos*100:.1f}% lefedettség", "#2563eb"),
    ("Átlag Táv Belváros", f"{atlag_belvaros_km:.2f} km", "Közúti/gyalogos hálózat", "#059669"),
    ("Pontos Minta Medián Ár", fmt_huf(med_nm_ar_pontos), "Fajlagos eladási ár/m²", "#7c3aed"),
    ("Városrészek Száma", f"{df_pontos['varosresz'].nunique()} db", "Térbeli lefedettség", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kőbánya Ár-Hőtérképe (Kernel Density Heatmap)

**📌 Mit mutat a simított hőtérkép?**
A fajlagos árak térbeli sűrűsödését:
- **Meleg színek (sárga/narancs)**: Kőbánya alsó vasútállomás és a Szent László tér környéke, valamint az Óhegy-park zöldövezeti része alkotja a legmagasabb árfekvésű tömböket.
- **Hideg színek (kék/lila)**: A Hős utca környéke és a külső vágányok melletti sávok alacsonyabb árszintet mutatnak.

In [3]:
fig1 = px.scatter_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    color='nm_ar_huf',
    size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True, 'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[df_pontos['nm_ar_huf'].quantile(0.05), df_pontos['nm_ar_huf'].quantile(0.95)],
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='Kőbányai lakások térbeli elhelyezkedése és négyzetméterára (Carto Positron térképen)'
)
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

### 3. Térbeli Árprofil (Kelet-Nyugat és Észak-Dél)

**📌 Mit mutat a keresztmetszeti ábra?**
A kerületen átívelő ártrendeket:
- **Kelet-Nyugati irány**: Ahogy a belső Hungária körúttól távolodunk Kelet felé a perifériára, a négyzetméterárak fokozatosan mérséklődnek.
- **Észak-Déli irány**: Északon (Zugló mellett) magasabb az árszint, mint a déli ipari zónák felé haladva.

In [4]:
fig2 = px.density_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    z='nm_ar_huf',
    radius=22,
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='Fajlagos ársűrűségi hőtérkép (Density Map) Kőbányán'
)
fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
fig2.show()

# Hálózati távolság a Mázsa tértől vs Ár/m²
if 'tavolsag_mazsa_halozati_m' in df_pontos.columns:
    fig3 = px.scatter(
        df_pontos,
        x='tavolsag_mazsa_halozati_m',
        y='nm_ar_huf',
        color='varosresz',
        trendline='lowess',
        title='Hálózati séta távolság a Mázsa tértől (m) vs. Négyzetméterár (LOWESS trendvonallal)',
        labels={'tavolsag_mazsa_halozati_m': 'Mázsa tér hálózati távolság (méter)', 'nm_ar_huf': 'Ár / m² (HUF)'},
        template=PLOTLY_TEMPLATE
    )
    fig3.update_layout(height=480)
    fig3.show()

### 4. Interaktív Térképi Szűrő Pult

**📌 Mire használható a szűrő?**
Árkategória és alapterület szerint fókuszálhat a térképen lévő ingatlanokra, megfigyelve a különböző kategóriák térbeli tömörülését.

In [5]:
# Interaktív Többváltozós Térbeli Térkép (Plotly updatemenus)
map_sub = df[df['minta_garantalt_pontos'] == 1].copy()

map_vars = [
    ('nm_ar_huf', '1. Fajlagos Négyzetméterár (HUF/m²)', 'Viridis', ':.0f'),
    ('ar_millio_ft', '2. Kínálati Vételár (M Ft)', 'Plasma', ':.1f'),
    ('alapterulet_nm', '3. Lakás Alapterület (m²)', 'Blues', ':.0f'),
    ('szobaszam_osszes', '4. Szobaszám (db)', 'Purples', ':.1f'),
    ('allapot_kod', '5. Műszaki Állapot Kód (1-6)', 'Greens', ':.0f'),
    ('tavolsag_metro_halozati_m', '6. Metróállomás Hálózati Sétaút (m)', 'RdYlBu', ':.0f'),
    ('tavolsag_vasut_m', '7. Vasúti Pálya Légvonal / Zaj (m)', 'Spectral', ':.0f')
]

fig_map = go.Figure()
buttons = []

for i, (col, label, colscale, fmt) in enumerate(map_vars):
    sub_f = px.scatter_map(
        map_sub,
        lat='geokodolt_lat',
        lon='geokodolt_lon',
        color=col,
        size='alapterulet_nm',
        size_max=12,
        hover_name='cim_teljes',
        hover_data={'varosresz': True, col: fmt, 'alapterulet_nm': ':.0f'},
        color_continuous_scale=colscale,
        zoom=12.2,
        center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
        map_style='carto-positron'
    )
    tr = sub_f.data[0]
    tr.visible = (i == 0)
    tr.name = label
    fig_map.add_trace(tr)
    
    vis = [j == i for j in range(len(map_vars))]
    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': vis}, {'title': f'Interaktív Térbeli Eloszlás: {label} (N={len(map_sub)})'}]
    ))

fig_map.update_layout(
    title=f'Interaktív Térbeli Eloszlás: {map_vars[0][1]} (N={len(map_sub)})',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    map_style='carto-positron',
    map_zoom=12.2,
    map_center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    height=580,
    margin={"r":0,"t":50,"l":0,"b":0}
)
fig_map.show()